# Exploratory Data Analysis: UCI Heart Disease Dataset
## 1. Project Overview


This notebook conducts an Exploratory Data Analysis (EDA) on the Cleveland Heart Disease Dataset. The goal is to analyze clinical and demographic features to understand the markers associated with cardiovascular disease. This dataset is a standard benchmark in medical machine learning, offering a mix of categorical, ordinal, and continuous physiological data.

### 1.1 Install Requirements

In [ ]:
!pip install matplotlib seaborn

In [ ]:
!pip install ucimlrepo

## 2. Fetch Data

This dataset contains 303 instances and 13 features, with a target variable num indicating the presence or absence of disease.

In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
heart_disease = fetch_ucirepo(id=45) 
  
# data (as pandas dataframes) 
X = heart_disease.data.features 
y = heart_disease.data.targets 
  
# metadata 
print(heart_disease.metadata) 
  
# variable information 
print(heart_disease.variables) 


## 3. Data Overview
Key clinical variables include:

- Vital Signs: trestbps (resting blood pressure), chol (serum cholesterol), and thalach (maximum heart rate).

- Cardiac Markers: oldpeak (ST depression induced by exercise) and slope (the slope of the peak exercise ST segment).

- Diagnostic Test Results: ca (number of major vessels colored by fluoroscopy) and thal (thalassemia diagnosis).

Target Distribution: The num target is graded, but for binary classification, we can treat 0 as "Healthy" and 1-4 as "Presence of Disease."

- No Heart Disease (num=0): 164 cases
- Heart Disease Present (num=1-4): 139 cases

In [ ]:
import pandas as pd

# Show first few rows
print("Features (X) head:")
print(X.head())

print("\nTarget (y) head:")
print(y.head())

# Check shape
print("\nShape of X:", X.shape)
print("Shape of y:", y.shape)

# Check column names and types
print("\nColumn info:")
print(X.info())

# Check target distribution
print("\nTarget distribution:")
print(y.value_counts())


## 4. Summary Statistics

With the dataset loaded, we perform a high-level statistical overview to identify data types, distributions, and potential anomalies.

### 4.1 Feature Statistics

Initial observations:

- Age Range: The study covers patients from 29 to 77 years old, with a mean age of approximately 54 years.
- Cholesterol (chol): There is significant variance here ($std \approx 51.7$), with a maximum value of 564 mg/dl, which may be an outlier worth investigating.
- Heart Rate (thalach): The average maximum heart rate achieved during exercise is 149 bpm, ranging from 71 to 202 bpm.

### 4.2 Data Integrity: Missing Values
The dataset is largely complete, but there are minor gaps in key diagnostic features:

- ca (Major vessels): 4 missing values.

- thal (Thallium test): 2 missing values.

Since the missing values account for less than 2% of the total records, we can address these using mode imputation (for categorical/ordinal data like ca and thal) or by removing the specific rows if they significantly skew the distribution.

### 4.3 Preliminary Data Challenges
- Outliers: The maximum cholesterol (564) and ST depression (oldpeak of 6.2) are significantly higher than their respective 75th percentiles.

- Scaling: Features like chol and thalach have much larger numerical ranges than oldpeak or sex. Feature scaling.

In [ ]:
# Summary statistics for features
print("\nSummary statistics:")
print(X.describe())

# Check for missing values
print("\nMissing values per column:")
print(X.isnull().sum())


## 4. Outlier Detection and Data Cleaning

Before proceeding with correlation analysis, we must address the anomalies identified in the summary statistics—specifically the high maximum values in chol (serum cholesterol) and oldpeak (ST depression).

### 4.1 Visualizing Outliers

We use boxplots to identify "flyers" (data points that fall outside $1.5 \times IQR$).

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set up the figure for key numerical features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(ax=axes[0], x=X['chol'], color='skyblue').set_title('Cholesterol Outliers')
sns.boxplot(ax=axes[1], x=X['oldpeak'], color='salmon').set_title('Oldpeak (ST Depression) Outliers')
sns.boxplot(ax=axes[2], x=X['trestbps'], color='lightgreen').set_title('Resting BP Outliers')

plt.show()

**Observations:**

- Cholesterol: Several patients have levels above 400 mg/dl, including one extreme case at 564 mg/dl.

- Oldpeak: Values above 4.0 are rare and represent significant ST depression, which is a strong clinical indicator of coronary artery disease.

### 4.2 Handling Missing Values
Since we only have 6 missing values total (4 in ca and 2 in thal), we can use mode imputation. These are categorical/ordinal features, so the most frequent value is the safest.

In [ ]:
# Impute missing values with the mode
X_cleaned = X.copy()
X_cleaned['ca'] = X_cleaned['ca'].fillna(X_cleaned['ca'].mode()[0])
X_cleaned['thal'] = X_cleaned['thal'].fillna(X_cleaned['thal'].mode()[0])

# Verify cleaning
print("Remaining missing values:")
print(X_cleaned.isnull().sum().sum())

### 4.3 Save cleaned data to csv

In [ ]:
df_cleaned = pd.concat([X_cleaned, y], axis=1)

# Save to CSV
df_cleaned.to_csv('heart_disease_cleaned.csv', index=False)

In [ ]:
# Load the cleaned data
df = pd.read_csv('heart_disease_cleaned.csv')

# Verify the first few rows and shape
print(f"Dataset Shape: {df.shape}")
df.head()

## 5. Correlation Analysis: Identifying Clinical Drivers

Explore the relationships between our variables. A correlation matrix to help us identify which clinical markers (like chest pain or maximum heart rate) move in tandem with the presence of heart disease.

### 5.1 Generating the Correlation Heatmap

Values closer to $1$ or $-1$ indicate a strong relationship, while values near $0$ suggest no linear correlation.

**Strongest Positive Correlations with Disease (num)**
- ca (0.52): The number of major vessels colored by fluoroscopy is the strongest positive indicator. As ca increases, so does the presence of heart disease.

- thal (0.51): The thallium stress test results show a high correlation, indicating its critical role in diagnosis.

- oldpeak (0.50): ST depression induced by exercise is a major diagnostic signal.

- cp (0.41): Chest pain type shows a significant relationship, particularly when patients report asymptomatic or non-anginal pain.

**Strongest Negative Correlations**

- thalach (-0.42): Maximum heart rate achieved shows a strong inverse relationship with the target. Higher cardiovascular fitness (the ability to reach a higher heart rate under stress) is associated with a lower risk of disease.

- thalach vs. age (-0.39): This confirms the biological reality that maximum heart rate capacity naturally declines as patients age.

**Secondary Observations**
- exang (0.40): Exercise-induced angina is a reliable predictor of the target variable.

- Weak Correlations: Fasting blood sugar (fbs at 0.06) and serum cholesterol (chol at 0.07) show surprisingly weak linear correlations with the target in this specific dataset, suggesting they may act as secondary factors rather than direct predictors.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate the correlation matrix
corr_matrix = df.corr()

# Set up the matplotlib figure
plt.figure(figsize=(12, 10))

# Generate a mask for the upper triangle (optional, for cleaner look)
import numpy as np
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Draw the heatmap
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap='RdBu_r', 
            center=0, linewidths=.5, cbar_kws={"shrink": .8})

plt.title('Feature Correlation Heatmap')
plt.show()

## 6. Visualizing Key Predictors
To better understand the distributions of our most influential features (oldpeak and thalach), we will generate comparison plots across the different target categories.

In [ ]:

# Distribution of Max Heart Rate vs. Heart Disease
plt.figure(figsize=(10, 6))
sns.violinplot(x='num', y='thalach', data=df, palette='coolwarm')
plt.title('Max Heart Rate (thalach) Distribution by Disease Severity')
plt.xlabel('Disease Severity (0=Healthy, 1-4=Sick)')
plt.ylabel('Max Heart Rate')
plt.show()

### 6.1 Multivariate Analysis: Pairplot of Key Predictors

Based on our correlation heatmap, the features ca, thal, oldpeak, and thalach show the strongest relationships with the target variable num. By plotting these in a matrix, we can observe:

- Diagonal Plots: The distribution (density) of each feature for healthy vs. diseased patients.

- Scatter Plots: How two features (e.g., Max Heart Rate vs. ST Depression) interact to separate the classes.


The Pairplot of the top four predictors (ca, thal, oldpeak, thalach) reveals how these variables separate healthy patients from those with heart disease:

- Class Separation: In the diagonal KDE plots, healthy patients (num=0, purple) form distinct, sharp peaks—particularly in oldpeak (near 0) and ca (near 0). Patients with heart disease (levels 1-4) show much broader, flatter distributions.

- Threshold Effects: In the oldpeak vs. thalach scatter plot, a "danger zone" is visible: patients with an oldpeak > 2.0 and a thalach < 120 are almost exclusively classified with heart disease.

- Ordinal Barriers: Features like ca (number of vessels) act as discrete "checkpoints." Most healthy patients have 0 vessels colored, while the presence of even 1 or 2 vessels significantly increases the likelihood of a higher num classification.


In [ ]:
# Selecting the top 4 features plus the target
top_features = ['ca', 'thal', 'oldpeak', 'thalach', 'num']
df_subset = df[top_features]


sns.pairplot(df_subset, hue='num', palette='viridis', diag_kind='kde', plot_kws={'alpha': 0.6})
plt.suptitle('Pairplot of Top 4 Heart Disease Predictors', y=1.02)
plt.show()

# Older stuff

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for all numeric features
X.hist(bins=20, figsize=(12,10))
plt.suptitle("Feature Distributions")
plt.show()

# Correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(X.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()


In [ ]:
# Combine X and y for convenience
df = X.copy()
df['target'] = y

# Boxplots for numeric features vs target
numeric_cols = X.select_dtypes(include='number').columns
for col in numeric_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='target', y=col, data=df)
    plt.title(f'{col} vs Target')
    plt.show()


In [ ]:
categorical_cols = X.select_dtypes(include='object').columns
for col in categorical_cols:
    print(f"\nValue counts for {col}:")
    print(X[col].value_counts())
    sns.countplot(x=col, hue='target', data=df)
    plt.show()
